# Section 04: Analyzing and Optimizing Processes

**Companion notebook for**: `04-analyzing-optimizing.html`

## Learning Objectives
- Build CI/CD pipelines with Cloud Build
- Configure disaster recovery strategies
- Analyze cost optimization opportunities
- Work with billing APIs and Recommender
- Design Cloud Deploy delivery pipelines

## Prerequisites
- A Google Cloud project with billing enabled
- Python 3.10+

---
## Setup

In [ ]:
%pip install -q google-cloud-billing google-cloud-recommender

In [ ]:
import os, json
PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'your-project-id')
print(f'Project: {PROJECT_ID}')
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

---
## Section 1: CI/CD Pipeline Configuration

Generate Cloud Build and Cloud Deploy configurations.

In [ ]:
cloudbuild_yaml = '''# cloudbuild.yaml
steps:
  # Unit tests
  - name: 'python:3.12'
    entrypoint: 'bash'
    args: ['-c', 'pip install -r requirements.txt && pytest tests/']

  # Build Docker image
  - name: 'gcr.io/cloud-builders/docker'
    args: ['build', '-t', 'us-central1-docker.pkg.dev/$PROJECT_ID/repo/app:$SHORT_SHA', '.']

  # Push to Artifact Registry
  - name: 'gcr.io/cloud-builders/docker'
    args: ['push', 'us-central1-docker.pkg.dev/$PROJECT_ID/repo/app:$SHORT_SHA']

  # Deploy to Cloud Run
  - name: 'gcr.io/cloud-builders/gcloud'
    args:
      - 'run'
      - 'deploy'
      - 'my-app'
      - '--image=us-central1-docker.pkg.dev/$PROJECT_ID/repo/app:$SHORT_SHA'
      - '--region=us-central1'

options:
  logging: CLOUD_LOGGING_ONLY
'''
print(cloudbuild_yaml)

In [ ]:
cloud_deploy_yaml = '''# cloud-deploy pipeline
apiVersion: deploy.cloud.google.com/v1
kind: DeliveryPipeline
metadata:
  name: my-pipeline
serialPipeline:
  stages:
    - targetId: dev
      profiles: [dev]
    - targetId: staging
      profiles: [staging]
      strategy:
        canary:
          canaryDeployment:
            percentages: [25, 50, 75]
    - targetId: production
      profiles: [production]
      strategy:
        canary:
          canaryDeployment:
            percentages: [10, 25, 50]
'''
print('Cloud Deploy Pipeline Configuration')
print('=' * 50)
print(cloud_deploy_yaml)

---
## Section 2: Disaster Recovery Planning

Compare DR patterns and calculate error budgets.

In [ ]:
dr_patterns = [
    {'pattern': 'Cold Standby', 'rpo': '24h', 'rto': '24h+', 'cost': '$', 'setup': 'Backups in DR region, rebuild from IaC'},
    {'pattern': 'Warm Standby', 'rpo': '5-15min', 'rto': '30min-4h', 'cost': '$$', 'setup': 'Scaled-down replica, DB read replicas'},
    {'pattern': 'Hot Standby', 'rpo': '<1min', 'rto': '<5min', 'cost': '$$$', 'setup': 'Full replica, near-sync replication'},
    {'pattern': 'Active-Active', 'rpo': '~0', 'rto': '~0', 'cost': '$$$$', 'setup': 'Both regions serve traffic, global LB'},
]

print(f"{'Pattern':<20} {'RPO':<12} {'RTO':<14} {'Cost':<6} {'Setup'}")
print('-' * 90)
for dr in dr_patterns:
    print(f"{dr['pattern']:<20} {dr['rpo']:<12} {dr['rto']:<14} {dr['cost']:<6} {dr['setup']}")

print('\nDR Matching Guide:')
requirements = [
    ('RPO 24h, RTO 24h, budget-constrained', 'Cold Standby'),
    ('RPO 15min, RTO 1h, e-commerce', 'Warm Standby'),
    ('RPO <1min, RTO <5min, financial services', 'Hot Standby'),
    ('RPO ~0, RTO ~0, global SaaS', 'Active-Active'),
]
for req, pattern in requirements:
    print(f'  {req:<50} --> {pattern}')

In [ ]:
# Error budget calculator
def error_budget(slo_target, period_days=30):
    total_minutes = period_days * 24 * 60
    budget_minutes = total_minutes * (1 - slo_target)
    budget_hours = budget_minutes / 60
    return budget_minutes, budget_hours

print('Error Budget Calculator')
print('=' * 60)
slo_targets = [0.99, 0.999, 0.9995, 0.9999]
for target in slo_targets:
    mins, hrs = error_budget(target)
    print(f'  SLO {target*100:.2f}% --> {mins:.1f} min/month ({hrs:.2f} hours) of allowed downtime')

---
## Section 3: Cost Optimization Analysis

In [ ]:
cost_strategies = [
    {'strategy': 'Spot VMs', 'savings': '60-91%', 'commitment': 'None', 'risk': 'Preemption', 'best_for': 'Batch, CI/CD, fault-tolerant'},
    {'strategy': 'CUDs (1yr)', 'savings': 'Up to 37%', 'commitment': '1 year', 'risk': 'Lock-in', 'best_for': 'Steady-state production'},
    {'strategy': 'CUDs (3yr)', 'savings': 'Up to 57%', 'commitment': '3 years', 'risk': 'Long lock-in', 'best_for': 'Predictable workloads'},
    {'strategy': 'SUDs', 'savings': 'Up to 30%', 'commitment': 'None (auto)', 'risk': 'None', 'best_for': 'VMs running >25% month'},
    {'strategy': 'Right-sizing', 'savings': '10-40%', 'commitment': 'None', 'risk': 'Performance', 'best_for': 'Over-provisioned VMs'},
    {'strategy': 'Autoscaling', 'savings': 'Variable', 'commitment': 'None', 'risk': 'Cold starts', 'best_for': 'Variable traffic'},
    {'strategy': 'Scale to Zero', 'savings': 'Up to 100%', 'commitment': 'None', 'risk': 'Cold starts', 'best_for': 'Cloud Run, intermittent'},
    {'strategy': 'Storage Lifecycle', 'savings': '40-80%', 'commitment': 'None', 'risk': 'Retrieval cost', 'best_for': 'Aging data'},
]

print(f"{'Strategy':<20} {'Savings':<12} {'Commitment':<16} {'Best For'}")
print('-' * 75)
for s in cost_strategies:
    print(f"{s['strategy']:<20} {s['savings']:<12} {s['commitment']:<16} {s['best_for']}")

In [ ]:
# Cost optimization scenario analyzer
def analyze_cost(monthly_spend, vm_count, avg_utilization, traffic_pattern):
    recommendations = []
    potential_savings = 0
    
    if avg_utilization < 0.3:
        savings = monthly_spend * 0.25
        recommendations.append(f'Right-size VMs (avg util {avg_utilization:.0%}): save ~${savings:,.0f}/mo')
        potential_savings += savings
    
    if traffic_pattern == 'steady' and vm_count >= 5:
        savings = monthly_spend * 0.37
        recommendations.append(f'1-year CUDs for {vm_count} steady VMs: save ~${savings:,.0f}/mo')
        potential_savings += savings
    
    if traffic_pattern == 'bursty':
        recommendations.append('Enable autoscaling to handle traffic dips')
        potential_savings += monthly_spend * 0.15
    
    recommendations.append('Enable billing budget alerts at 50%, 90%, 100%')
    recommendations.append('Review Recommender insights monthly')
    
    return recommendations, potential_savings

scenarios = [
    ('Prod cluster', 15000, 20, 0.25, 'steady'),
    ('Dev environment', 5000, 10, 0.15, 'bursty'),
    ('Data pipeline', 8000, 8, 0.70, 'bursty'),
]

for name, spend, vms, util, pattern in scenarios:
    recs, savings = analyze_cost(spend, vms, util, pattern)
    print(f'\n{name} (${spend:,}/mo, {vms} VMs, {util:.0%} avg util, {pattern})')
    print(f'  Potential savings: ${savings:,.0f}/mo')
    for r in recs:
        print(f'  --> {r}')

---
## Section 4: Billing Budget Commands

In [ ]:
billing_cmds = [
    ('Create budget alert', '''gcloud billing budgets create \\
    --billing-account=01ABCD-234567-EFGH89 \\
    --display-name="Monthly Budget" \\
    --budget-amount=5000 \\
    --threshold-rule=percent=50 \\
    --threshold-rule=percent=90 \\
    --threshold-rule=percent=100'''),
    ('List recommendations', '''gcloud recommender insights list \\
    --insight-type=google.compute.instance.MachineTypeRecommender \\
    --project=PROJECT_ID --location=us-central1-a'''),
    ('Export billing to BQ', 'Configure in Console: Billing > Billing export > BigQuery export'),
]

for desc, cmd in billing_cmds:
    print(f'## {desc}')
    print(f'{cmd}\n')

---
## Section 5: Stakeholder Communication Matrix

In [ ]:
stakeholders = [
    ('Executive', 'TCO, risk reduction, competitive advantage', 'Business metrics, ROI', 'Quarterly business review'),
    ('Dev Team', 'Developer experience, CI/CD, reduced ops', 'Technical demos, migration path', 'Sprint reviews'),
    ('Security', 'Controls, audit, compliance certs', 'Defense-in-depth diagram', 'Security reviews'),
    ('Operations', 'Monitoring, alerting, IaC, incident response', 'Runbooks, dashboards', 'Weekly ops sync'),
    ('Finance', 'Budget, forecasting, chargeback', 'Cost dashboards, billing reports', 'Monthly cost review'),
]

print(f"{'Stakeholder':<14} {'Focus Areas':<40} {'Communication Style':<30} {'Cadence'}")
print('-' * 110)
for s in stakeholders:
    print(f'{s[0]:<14} {s[1]:<40} {s[2]:<30} {s[3]}')

---
## Summary

This notebook covered PCA Section 04:

1. **CI/CD** -- Cloud Build and Cloud Deploy configurations
2. **Disaster Recovery** -- DR patterns, RPO/RTO matching, error budgets
3. **Cost Optimization** -- Strategies, scenario analysis, savings calculator
4. **Billing** -- Budget alerts, Recommender, billing export
5. **Stakeholder Communication** -- Matrix by audience type

**Next**: Section 05 covers managing implementation.